# Field-read BusNet — private modules, synchrony binding, one bus

The model this notebook tests is the design from the discussion: keep the
**oscillator field** as the binder (cells of one object synchronise), but
replace the slot *competition* with per-module *reads* — each of six
**privately parameterised** modules (own GRU cell, own identity embedding)
computes its own phase-space query and attends over the field's cells with
it. Binding is still by synchrony (the groups a query can grab are the
groups the field formed); identity and specialisation live in parameters;
communication is the unchanged bus (write $m_j z_j^\top$, one wire, receive
through your own frame, echo cancelled). The head holds only the question.

| arm | question |
|---|---|
| `private` | the model |
| `shared-gru` | one shared cell: is private parameterisation doing anything? |
| `static` | learned fixed addresses, no phase dynamics: are computed addresses doing anything? |

Interventions at the end: `freeze` (no phase dynamics at test) and
`shuffle` (permute the module addresses) — the addresses-are-load-bearing check.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'main.py').exists())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import torch
import torch.nn as nn

from src.core.config import Config, TrainConfig, LoggingConfig, OptimConfig
from src.tasks.sort_of_clevr.config import SortOfClevrDataConfig
from src.tasks.sqoop.config import SqoopDataConfig
from src.tasks.sort_of_clevr.callbacks import AccuracyCallbackCfg, QtypeAccuracyCallbackCfg
from src.tasks.sqoop.callbacks.metrics import SqoopAccuracyCallbackCfg
from src.tasks import TASKS


In [2]:
# THE MODEL -- field binding + private phase-query reads + one bus.
#
# The oscillator field binds by synchrony exactly as in the canonical
# SyncNet: every cell of the feature map carries 16 unit 4-vectors that
# settle under rotation + conv coupling + feature stimulus, so cells of
# one object end up sharing a phase. The change is what happens next.
# Instead of slots COMPETING for cells (softmax over slots -> a partition,
# exchangeable modules), each of six PRIVATE modules computes its own
# phase-space query from its initial state and READS the field with it
# (softmax over cells). Identity comes from parameters (private GRU cell +
# a learned embedding), specialisation from what each module learns to
# tune its query to. Communication is the unchanged bus: write m_j z_j^T,
# one wire carries the sum, receive through your own frame, cancel your
# echo. The head holds only the question and listens; answer = head + prior.
#
# flags: private=False shares one GRU cell (the exchangeable control);
#        static=True freezes learned per-row addresses (no dynamics);
#        forward(..., phase_override='freeze'|'shuffle') for interventions.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

M, D, DM, MSG, T, DT = 6, 6, 96, 4, 8, 0.5          # modules, phase dim, state, message, bus steps
FCH, FG, FD, TF = 64, 16, 4, 8                       # field: channels, groups, osc dim, steps
TOK, HID, BETA = 64, 128, 8.0


def tang(z, v):
    return v - (v * z).sum(-1, keepdim=True) * z


class Field(nn.Module):
    """16 unit 4-vectors per cell; TF steps of rotate + couple + stimulus."""

    def __init__(self):
        super().__init__()
        C = FG * FD
        self.z_head = nn.Conv2d(FCH, C, 1)
        self.stim = nn.Conv2d(FCH, C, 1)
        self.J = nn.Conv2d(C, C, 5, padding=2, bias=False)
        nn.init.normal_(self.J.weight, std=0.02)
        self.omega = nn.Parameter(torch.randn(FG, FD, FD) * 0.1)

    def _norm(self, z):
        B, C, S, _ = z.shape
        return F.normalize(z.view(B, FG, FD, S, S), dim=2).view(B, C, S, S)

    def forward(self, f):
        z = self._norm(self.z_head(f))
        c = self.stim(f)
        A = (self.omega - self.omega.transpose(-1, -2)) * 0.1
        B, C, S, _ = z.shape
        for _ in range(TF):
            zg = z.view(B, FG, FD, S, S)
            rot = torch.einsum('kde,bkeij->bkdij', A, zg).reshape(B, C, S, S)
            dg = (self.J(z) + c).view(B, FG, FD, S, S)
            drv = (dg - (dg * zg).sum(2, keepdim=True) * zg).reshape(B, C, S, S)
            z = self._norm(z + rot + drv)
        return z.view(B, FG, FD, S * S).permute(0, 3, 1, 2)                 # (B, P, FG, FD)


class FieldReadBusNet(nn.Module):

    def __init__(self, img_size: int, q_size: int, answer_dim: int,
                 private: bool = True, static: bool = False):
        super().__init__()
        self.q_size, self.private, self.static, self.N = q_size, private, static, M + 1
        layers, cin, s = [], 3, img_size
        for _ in range(2):
            layers += [nn.Conv2d(cin, 48, 3, 2, 1), nn.GroupNorm(8, 48), nn.SiLU()]
            cin, s = 48, (s + 1) // 2
        layers += [nn.Conv2d(48, 48, 3, 1, 1), nn.GroupNorm(8, 48), nn.SiLU(), nn.Conv2d(48, FCH, 3, 1, 1)]
        self.enc = nn.Sequential(*layers)
        self.g_gamma, self.g_beta = nn.Linear(q_size, FCH), nn.Linear(q_size, FCH)
        self.gn = nn.GroupNorm(8, FCH)
        self.pos = nn.Parameter(0.02 * torch.randn(1, FCH, s, s))
        self.field = Field()
        # private identity: embedding + (below) one GRU cell per row
        self.e = nn.Parameter(torch.randn(M, DM) / DM ** 0.5)
        self.h_init = nn.Sequential(nn.Linear(q_size, 64), nn.GELU(), nn.Linear(64, M * DM))
        self.head_init = nn.Sequential(nn.Linear(q_size, 64), nn.GELU(), nn.Linear(64, DM))
        self.q_phi = nn.Linear(DM, FG * FD)                                 # the module's phase-space query
        self.to_tok = nn.Sequential(nn.LayerNorm(FCH), nn.Linear(FCH, TOK))
        self.f2g, self.f2b = nn.Linear(q_size, TOK), nn.Linear(q_size, TOK)
        self.tnorm = nn.LayerNorm(TOK)
        mk = lambda: nn.GRUCell(TOK + MSG * D, DM)
        self.cells = nn.ModuleList([mk() for _ in range(self.N)]) if private else mk()
        self.msg = nn.Linear(DM, MSG)
        g = torch.Generator().manual_seed(1234)
        self.register_buffer('ref', F.normalize(torch.randn(D - 1, D, generator=g), dim=-1))
        self.az = nn.Linear(FG * FD, D)                                     # query -> starting bus address
        if static:
            self.z_static = nn.Parameter(F.normalize(torch.randn(self.N, D), dim=-1))
        self.omega = nn.Parameter(torch.zeros(self.N))
        self.Kc = nn.Parameter(torch.ones(self.N, self.N))
        self.kmlp = nn.Sequential(nn.Linear(2 * DM, 64), nn.GELU(), nn.Linear(64, 1), nn.Tanh())
        self.zstim = nn.Linear(DM, D)
        self.A = nn.Parameter(0.1 * torch.randn(D, D))
        self.out = nn.Sequential(nn.Linear(DM + q_size, HID), nn.GELU(), nn.Linear(HID, answer_dim))
        self.prior = nn.Sequential(nn.Linear(q_size, HID), nn.GELU(), nn.Linear(HID, answer_dim))

    def _frame(self, z):
        vecs = [z]
        for k in range(D - 1):
            v = self.ref[k].to(z.dtype).expand_as(z)
            for u in vecs:
                v = v - (v * u).sum(-1, keepdim=True) * u
            vecs.append(F.normalize(v, dim=-1))
        return torch.stack(vecs, 2)

    def _receive(self, h, z):
        m = self.msg(h)
        bus = torch.einsum('bnD,bnd->bDd', m, z)
        Fr = self._frame(z)
        r = torch.einsum('bDd,bnad->bnDa', bus, Fr) - torch.einsum('bnD,bnd,bnad->bnDa', m, z, Fr)
        return r.flatten(2) / float(self.N)

    def _zstep(self, z, h):
        B = h.shape[0]
        A = self.A - self.A.t()
        A = A / (A.norm() / math.sqrt(2) + 1e-6)
        vel = self.omega.to(z.dtype)[None, :, None] * torch.einsum('de,bne->bnd', A, z)
        hi = h.unsqueeze(2).expand(B, self.N, self.N, DM)
        hj = h.unsqueeze(1).expand(B, self.N, self.N, DM)
        kap = self.kmlp(torch.cat([hi, hj], -1)).squeeze(-1)
        vel = vel + tang(z, torch.einsum('bij,bjd->bid', self.Kc.to(z.dtype)[None] * kap, z))
        vel = vel + tang(z, self.zstim(h))
        return F.normalize(z + DT * vel, dim=-1)

    def forward(self, images, q, phase_override=None):
        B = images.shape[0]
        q = q.float()
        f = self.enc(images)
        f = f * (1 + self.g_gamma(q))[..., None, None] + self.g_beta(q)[..., None, None]
        f = self.gn(f) + self.pos
        Zt = self.field(f)                                                  # binding by synchrony
        feats = f.flatten(2).transpose(1, 2)
        h = torch.cat([self.h_init(q).reshape(B, M, DM) + self.e[None],
                       self.head_init(q).unsqueeze(1)], 1)
        phi = F.normalize(self.q_phi(h[:, :M]).view(B, M, FG, FD), dim=-1)  # each module's query
        for _ in range(3):                                                  # refine: query -> read -> move
            attn = F.softmax(BETA * torch.einsum('bmgd,bpgd->bmp', phi, Zt) / FG, dim=-1)   # over CELLS
            phi = F.normalize(torch.einsum('bmp,bpgd->bmgd', attn, Zt), dim=-1)
        X = torch.einsum('bmp,bpf->bmf', attn, feats)
        X = self.tnorm(self.to_tok(X) * (1 + self.f2g(q)).unsqueeze(1) + self.f2b(q).unsqueeze(1))
        X = torch.cat([X, torch.zeros(B, 1, TOK, device=X.device, dtype=X.dtype)], 1)
        if self.static:
            z = F.normalize(self.z_static, dim=-1)[None].expand(B, -1, -1).to(X.dtype)
        else:
            zs = F.normalize(self.az(phi.flatten(2)), dim=-1)               # address from the query
            if phase_override == 'shuffle':
                zs = zs[:, torch.randperm(M, device=X.device)]
            zh = F.normalize(torch.randn(B, 1, D, device=X.device, dtype=X.dtype), dim=-1)
            z = torch.cat([zs, zh], 1)
        for _ in range(T):
            r = self._receive(h, z)
            inp = torch.cat([X, r], -1)
            if self.private:
                h = torch.stack([self.cells[k](inp[:, k], h[:, k]) for k in range(self.N)], 1)
            else:
                h = self.cells(inp.reshape(B * self.N, -1), h.reshape(B * self.N, DM)).reshape(B, self.N, DM)
            if not self.static and phase_override != 'freeze':
                z = self._zstep(z, h)
        logits = self.out(torch.cat([h[:, M], q], -1)) + self.prior(q)
        return {'logits': logits, 'read_attn': attn, 'z': z}


class OnTask(nn.Module):
    """Adapts the core to a task batch: SoC questions are 18-d vectors,
    SQOOP questions are 3 ints one-hot encoded to 120-d."""

    def __init__(self, core: FieldReadBusNet, kind: str):
        super().__init__()
        self.core, self.kind = core, kind

    def forward(self, batch, **kw):
        if self.kind == 'soc':
            q = batch['questions'].float()
        else:
            q = F.one_hot(batch['questions'].long(), 40).float().flatten(1)
        ov = kw.get('phase_override')
        return self.core(batch['images'], q, phase_override=ov)


In [3]:
from accelerate import Accelerator
from accelerate.utils import ProjectConfiguration

from src import build_dataloaders, build_optim, build_lr_scheduler, build_loss_fn, build_callbacks
from src.training import Trainer
from src.training.utils import set_seed


def train(model: nn.Module, cfg: Config, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    set_seed(cfg.train.seed)
    accelerator = Accelerator(
        mixed_precision=cfg.train.mixed_precision,
        gradient_accumulation_steps=cfg.train.grad_accum,
        project_config=ProjectConfiguration(project_dir=out_dir),
    )
    try:
        optimiser = build_optim(model, cfg.optim)
        trainer = Trainer(
            cfg=cfg, out_dir=out_dir, logger=None, model=model,
            dataloaders=build_dataloaders(cfg, str(accelerator.device)),
            optimiser=optimiser,
            scheduler=build_lr_scheduler(optimiser, cfg.train.n_steps, cfg.optim),
            accelerator=accelerator, callbacks=build_callbacks(cfg),
            loss_fn=build_loss_fn(cfg),
        )
        return trainer.train(), trainer
    finally:
        accelerator.end_training()


def run_arms(mk_model, cfg, out_root, arms):
    results, models = {}, {}
    for name, kw in arms.items():
        model = mk_model(**kw)
        print(f'\n===== {name}  ({sum(p.numel() for p in model.parameters()):,} params) =====')
        results[name], _ = train(model, cfg, str(out_root / name.replace('/', '_')))
        models[name] = model
    return results, models


def intervene(model, loader, n_batches=4):
    model.eval(); out = {}
    with torch.no_grad():
        for ov in [None, 'freeze', 'shuffle']:
            n = c = 0
            for i, b in enumerate(loader):
                if i == n_batches: break
                p = model(b, phase_override=ov)['logits'].argmax(-1)
                c += (p == b['answers']).sum().item(); n += len(p)
            out[ov or 'none'] = c / max(n, 1)
    return out


ARMS = {
    'private':    dict(),
    'shared-gru': dict(private=False),
    'static':     dict(static=True),
}
STEPS = 10_000                     # quick test; the screens used 100k


In [4]:
OUT = ROOT / 'notebooks' / 'outputs' / 'fieldread_dev'

soc_ds = SortOfClevrDataConfig(
    name='sort_of_clevr', seed=1, root=str(ROOT / 'data'), dir='sort-of-clevr-notebook-dev',
    train_size=20_000, test_size=1000, img_size=75, obj_size=5, nb_questions=10, t_subtype=-1,
)
if not (Path(soc_ds.root) / soc_ds.dir).exists():
    TASKS['sort_of_clevr'].prepare(soc_ds)

def mk_cfg(ds, callbacks):
    return Config(
        train=TrainConfig(seed=0, n_steps=STEPS, train_bs=256, val_bs=1024,
                          early_stop_metric='loss', early_stop_big_is_better=False,
                          early_stop_patience=10**6, early_stop_min_delta=0.0,
                          mixed_precision='bf16', compile_model=False,
                          grad_accum=1, grad_clip=1.0, loader_mode='gpu_cached', num_workers=0),
        logging=LoggingConfig(eval_log_interval=500, train_log_interval=100,
                              info_metrics=['loss', 'accuracy'], save_best=False),
        optim=OptimConfig(optimiser='adamw', lr=3e-4, weight_decay=0.01,
                          lr_scheduler='warmup_cosine', lr_scheduler_params={'warmup_steps': 300}),
        dataset=ds, callbacks=callbacks,
    )

soc_cfg = mk_cfg(soc_ds, [AccuracyCallbackCfg(), QtypeAccuracyCallbackCfg()])
soc_results, soc_models = run_arms(
    lambda **kw: OnTask(FieldReadBusNet(75, 18, 10, **kw), 'soc'),
    soc_cfg, OUT / 'soc', ARMS,
)



===== private  (675,873 params) =====
Starting training
Model: OnTask
Parameters: total=675873
Config:
train:
  seed: 0
  n_steps: 10000
  train_bs: 256
  val_bs: 1024
  early_stop_metric: loss
  early_stop_big_is_better: false
  early_stop_patience: 1000000
  early_stop_min_delta: 0.0
  mixed_precision: bf16
  compile_model: false
  grad_accum: 1
  grad_clip: 1.0
  loader_mode: gpu_cached
  num_workers: 0
logging:
  eval_log_interval: 500
  train_log_interval: 100
  info_metrics:
  - loss
  - accuracy
  save_best: false
wandb:
  enabled: false
  project_name: null
  entity: null
  tags: []
  run_name: null
optim:
  optimiser: adamw
  lr: 0.0003
  weight_decay: 0.01
  lr_scheduler: warmup_cosine
  lr_scheduler_params:
    warmup_steps: 300
dataset:
  name: sort_of_clevr
  root: /home/nik/workspace/ImperialWork/msc_project/SyncNetProject/data
  dir: sort-of-clevr-notebook-dev
  seed: 1
  train_size: 20000
  test_size: 1000
  img_size: 75
  obj_size: 5
  nb_questions: 10
  t_subtype: -1

In [ ]:
FLOORS = {'accuracy': 0.492, 'binary_accuracy': 0.433, 'ternary_accuracy': 0.538}

print(f'{'arm':<12}' + ''.join(f'{k.split('_')[0]:>10}' for k in FLOORS) + f'{'freeze':>9}{'shuffle':>9}')
soc_test_loader = build_dataloaders(soc_cfg, 'cuda' if torch.cuda.is_available() else 'cpu')[2]
for name in ARMS:
    r = soc_results[name]
    row = ''.join(f'{r[f'callbacks/{k}'] - v:>+10.3f}' for k, v in FLOORS.items())
    iv = intervene(soc_models[name], soc_test_loader)
    row += f'{iv['none'] - iv['freeze']:>+9.3f}{iv['none'] - iv['shuffle']:>+9.3f}'
    print(f'{name:<12}' + row)

print('\naccuracies are deltas above the question-only floors; freeze / shuffle are')
print('accuracy drops under the intervention (positive = the phases are load-bearing).')
print('read collapse check: if private ~ shared-gru everywhere, identity bought nothing.')


SyntaxError: f-string: unmatched '[' (553797589.py, line 6)

In [ ]:
# SQOOP at rhs=18, dev sizes (full protocol is 1,080,000 / 100k steps).
# First run generates the dataset (~6 min single-core at these sizes).
sq_ds = SqoopDataConfig(
    name='sqoop', seed=0, root=str(ROOT / 'data'), dir='sqoop-notebook-dev-rhs18',
    train_size=103_680, test_size=12_240, rhs_variety=18,
)
sq_cfg = mk_cfg(sq_ds, [SqoopAccuracyCallbackCfg()])
sq_results, sq_models = run_arms(
    lambda **kw: OnTask(FieldReadBusNet(64, 120, 2, **kw), 'sqoop'),
    sq_cfg, OUT / 'sqoop', ARMS,
)


In [ ]:
print(f'{'arm':<12}{'train-test':>11}{'test':>8}{'freeze':>9}{'shuffle':>9}   (floor = .500)')
sq_test_loader = build_dataloaders(sq_cfg, 'cuda' if torch.cuda.is_available() else 'cpu')[2]
for name in ARMS:
    r = sq_results[name]
    iv = intervene(sq_models[name], sq_test_loader)
    print(f'{name:<12}{'':>11}{r['callbacks/accuracy']:>8.3f}'
          f'{iv['none'] - iv['freeze']:>+9.3f}{iv['none'] - iv['shuffle']:>+9.3f}')

print('\ntest is on held-out pairs (test_unseen). Expect chance at 3k steps -- SQOOP')
print('has no easy subtask, and every pixel front end so far fails its memorisation')
print('gate; the run documents whether this one is different, and the read_attn')
print('entropy in the aux outputs shows whether the private queries differentiated.')
